# Adicionar modelo de resposta manualmente

Notebook **separado** de `ler_dados_pst.ipynb` — aquele lê o `.pst` exportado
do Outlook inteiro; este serve para o caso menor: acrescentar (ou corrigir)
UM modelo de resposta direto em `modelos_resposta_chunks.xlsx`, sem precisar
reexportar e reprocessar a caixa de e-mail inteira.

Gera linhas no MESMO formato que `ler_dados_pst.ipynb` produziria — mesma
regra de chunking (`dividir`, `CHUNK_SIZE=1200`), mesmo cabeçalho
("Modelo de resposta padrao do suporte..."), mesmo `modelo_id` sequencial
(`MODxxxx`) — para que o loader de ingestão
(`app/ingestion/loaders/xlsx_modelos_resposta.py`) não veja diferença entre
uma linha vinda do export e uma escrita aqui.

**Fluxo:**
1. Rode as células até "estado atual do arquivo" para ver o que já existe.
2. Escreva os modelos novos num arquivo JSON (copie de
   `novos_modelos.exemplo.json`, salve como `novos_modelos.json` ao lado do
   notebook) e rode a célula que o lê.
3. Rode o resto: valida, gera as linhas, confira a prévia, faz backup do
   arquivo atual e sobrescreve `modelos_resposta_chunks.xlsx`.
4. Fora do notebook: `python -m scripts.ingest email_modelos`.

**Atenção — reingestão é do arquivo inteiro, não incremental**: `ingest_file`
apaga por `source_path` antes de reindexar (é o que mantém a reingestão
idempotente mesmo quando um arquivo encolhe — ver `app/ingestion/pipeline.py`).
Rodar o passo 4 reprocessa TODOS os modelos do arquivo, não só o que este
notebook acabou de adicionar. Local e sem custo de API (o embedding roda no
HuggingFace local do projeto), então é rápido, mas vale saber.

**Duplicação deliberada, não descuido**: a função `dividir()` logo abaixo é
uma CÓPIA da de `ler_dados_pst.ipynb` (seção 6a), não um import — notebook
não é módulo importável sem ferramenta extra (`nbimporter`), que este projeto
não tem só por causa de uma função. Se `CHUNK_SIZE`/`CHUNK_OVERLAP`/`dividir`
mudarem lá, replique aqui à mão — senão os dois arquivos passam a chunkar
diferente sem que nada avise.

In [1]:
from __future__ import annotations

import shutil
from datetime import datetime
from pathlib import Path

import pandas as pd

# Mesmo arquivo que a ingestão lê (data/raw/<assunto>/ = pasta que
# `python -m scripts.ingest <assunto>` processa — ver app/ingestion/pipeline.py).
CAMINHO_XLSX = Path("../../data/raw/email_modelos/modelos_resposta_chunks.xlsx").resolve()

# FORA de data/raw/ de propósito: `ingest_assunto` varre a pasta do assunto
# inteira com `rglob("*")` — recursivo, sem exceção para pasta com nome
# começando em ponto. Um backup salvo DENTRO de data/raw/email_modelos/
# seria descoberto como arquivo novo na próxima ingestão e voltaria a ser
# indexado, inclusive conteúdo já removido/corrigido numa edição anterior.
PASTA_BACKUP = Path(".bak").resolve()

assert CAMINHO_XLSX.exists(), f"não encontrado: {CAMINHO_XLSX}"
print("arquivo:", CAMINHO_XLSX)

arquivo: C:\Users\henrique.cordeiro\Desktop\PPP\ia-agent-puc-digital\data\raw\email_modelos\modelos_resposta_chunks.xlsx


## A mesma lógica de chunking de `ler_dados_pst.ipynb` (seção 6a)

In [2]:
CHUNK_SIZE      = 1200   # caracteres por chunk (~300 tokens)
CHUNK_OVERLAP   = 150
MIN_CHARS_CHUNK = 80     # descarta ruído


def dividir(texto, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    texto = (texto or "").strip()
    if len(texto) <= size:
        return [texto] if texto else []
    chunks, ini = [], 0
    while ini < len(texto):
        fim = min(ini + size, len(texto))
        if fim < len(texto):
            meio = ini + size // 2
            corte = max(texto.rfind("\n\n", meio, fim),
                        texto.rfind(". ", meio, fim),
                        texto.rfind("\n", meio, fim))
            if corte > ini:
                fim = corte + 1
        chunks.append(texto[ini:fim].strip())
        if fim >= len(texto):
            break
        ini = max(fim - overlap, ini + 1)
    return [c for c in chunks if len(c) >= MIN_CHARS_CHUNK]

## Estado atual do arquivo

In [3]:
xl = pd.ExcelFile(CAMINHO_XLSX)
df_modelos = xl.parse("modelos")
df_chunks = xl.parse("chunks")

print(f"modelos hoje: {len(df_modelos)} | chunks hoje: {len(df_chunks)}")
# So para conferencia visual — a celula de Geracao, mais abaixo, relê o
# disco de novo na hora de calcular o proximo modelo_id (nao usa este
# df_modelos/df_chunks), entao rodar esta celula fora de ordem e' seguro.

# "MOD0001" -> 1. Fatia fixa (prefixo "MOD" tem 3 letras) em vez de
# str.removeprefix: funciona em qualquer versão do pandas instalada.
_max_num = df_modelos["modelo_id"].str.slice(3).astype(int).max() if len(df_modelos) else 0
print(f"próximo modelo_id livre: MOD{_max_num + 1:04d}")

df_modelos[["modelo_id", "assunto", "n_ocorrencias", "redirecionado"]].tail(5)

modelos hoje: 184 | chunks hoje: 203
próximo modelo_id livre: MOD0185


,modelo_id,assunto,n_ocorrencias,redirecionado
179,MOD0180,Atestado de matrícula - [RA],3,False
180,MOD0181,RES: Declaração de estudante/ matrícula,3,False
181,MOD0182,Duvida sobre curso dupla pos,1,False
182,MOD0183,Esqueci minha senha,1,False
183,MOD0184,Nota de corte,1,False


## Adicione os modelos novos — por arquivo JSON

Escreva os modelos novos num arquivo JSON e aponte `CAMINHO_JSON` para ele.
A célula abaixo lê esse arquivo e monta `NOVOS_MODELOS` no **mesmo formato de
dict** de antes — as células seguintes (Validação, Geração, Gravação) não
mudam.

Por que JSON em vez do `input()` anterior: texto de modelo tem vários
parágrafos, aspas e `{{campo}}` — colar isso linha a linha num prompt é
frágil. Num arquivo você edita com calma, versiona junto num commit, revisa
o diff e roda de novo se algo falhar na validação, sem redigitar nada.

**Formato** — uma lista de objetos (veja `novos_modelos.exemplo.json` ao lado
do notebook):

```json
[
  {
    "assunto": "Nota de corte",
    "texto_modelo": "Olá, {{nome}}! Bom dia!\n\nNas disciplinas ...\n\nAtenciosamente,",
    "slots": ["nome"],
    "assuntos_variantes": ["RES: Nota de corte", "ENC: nota minima"],
    "redirecionado": false
  }
]
```

Campos:

- **`assunto`** (obrigatório) — assunto típico do e-mail que este modelo responde.
- **`texto_modelo`** (obrigatório) — corpo da resposta. Use `\n` para quebra
  de linha no JSON; `{{campo}}` para o que varia por aluno (mesma convenção
  do resto da base).
- **`slots`** (opcional) — lista com os nomes dos `{{campo}}` usados
  (`["nome", "valor"]`). Omita ou `null` se o modelo não tem campo variável.
  Aceita também string `"nome; valor"`.
- **`assuntos_variantes`** (opcional) — outras formas do mesmo assunto
  (`RES:`/`RE:`/`ENC:`/etc.). Omita → usa só o próprio `assunto`.
- **`redirecionado`** (opcional, default `false`) — `true` se a resposta só
  encaminha o aluno para outro contato, sem resolver aqui.

`n_ocorrencias` / `n_threads` são ignorados se vierem no JSON e ficam em `1`:
modelo escrito à mão não tem histórico medido, e esse número pode virar sinal
de confiança em ranking futuro — não infle (ver `ler_dados_pst.ipynb`).


In [4]:
import json

# Arquivo com os modelos novos. Fica ao lado do notebook (fora de data/raw/
# — mesma razão do backup: rglob da ingestao varreria e indexaria um .json
# solto na pasta do assunto). Troque o nome se estiver usando outro arquivo.
CAMINHO_JSON = Path("novos_modelos.json").resolve()

assert CAMINHO_JSON.exists(), (
    f"não encontrado: {CAMINHO_JSON}\n"
    "Crie o arquivo (veja novos_modelos.exemplo.json ao lado do notebook) "
    "ou ajuste CAMINHO_JSON."
)

_bruto = json.loads(CAMINHO_JSON.read_text(encoding="utf-8"))
if not isinstance(_bruto, list):
    raise TypeError(
        f"{CAMINHO_JSON.name}: raiz deve ser uma LISTA de objetos, veio "
        f"{type(_bruto).__name__}. Envolva num `[ ... ]` mesmo que seja um só."
    )

_CHAVES_ACEITAS = {
    "assunto", "texto_modelo", "slots", "assuntos_variantes", "redirecionado",
}

NOVOS_MODELOS = []
for _i, _m in enumerate(_bruto):
    _rot = f"{CAMINHO_JSON.name}[{_i}]"
    if not isinstance(_m, dict):
        raise TypeError(f"{_rot}: esperava objeto, veio {type(_m).__name__}.")

    # Chave desconhecida quase sempre é typo ("assunto_variantes", "redirecao")
    # — barra agora em vez de o campo ser silenciosamente ignorado.
    _extra = set(_m) - _CHAVES_ACEITAS
    if _extra:
        raise ValueError(
            f"{_rot}: chave(s) não reconhecida(s): {sorted(_extra)}. "
            f"Aceitas: {sorted(_CHAVES_ACEITAS)}. "
            "(n_ocorrencias/n_threads não entram aqui — modelo à mão fica em 1.)"
        )
    for _obrig in ("assunto", "texto_modelo"):
        if not str(_m.get(_obrig) or "").strip():
            raise ValueError(f"{_rot}: '{_obrig}' obrigatório e não pode ser vazio.")

    NOVOS_MODELOS.append({
        "assunto": _m["assunto"],
        "texto_modelo": _m["texto_modelo"],
        "slots": _m.get("slots"),
        "assuntos_variantes": _m.get("assuntos_variantes"),
        "redirecionado": bool(_m.get("redirecionado", False)),
        # Escrito a mao: sem historico medido -> 1 (ver markdown acima).
        "n_ocorrencias": 1,
        "n_threads": 1,
    })

print(f"{CAMINHO_JSON.name}: {len(NOVOS_MODELOS)} modelo(s) na fila:")
for _m in NOVOS_MODELOS:
    _var = _m["assuntos_variantes"] or []
    _extra = f" (+{len(_var)} variante(s))" if isinstance(_var, list) and _var else ""
    print(f"  - {_m['assunto']!r}{_extra} — {len(_m['texto_modelo'])} chars"
          + ("  [redirecionado]" if _m["redirecionado"] else ""))


novos_modelos.json: 13 modelo(s) na fila:
  - 'Alteração de e-mail cadastrado' (+3 variante(s)) — 481 chars
  - 'Envio de atividade no Canvas com prazo expirado' (+2 variante(s)) — 525 chars
  - 'Envio de tarefas com atraso por motivo médico' (+1 variante(s)) — 449 chars  [redirecionado]
  - 'Visualizar a grade curricular do curso' (+2 variante(s)) — 416 chars
  - 'Recuperação de senha e acesso à conta institucional' (+3 variante(s)) — 454 chars  [redirecionado]
  - 'Consultar notas de semestres anteriores' (+1 variante(s)) — 413 chars
  - 'Suporte do Canvas para navegador não suportado' (+1 variante(s)) — 622 chars
  - 'Pontuação de Atividades Complementares (AACC)' (+1 variante(s)) — 469 chars  [redirecionado]
  - 'Trancamento de disciplina ou do curso' (+4 variante(s)) — 744 chars  [redirecionado]
  - 'Reagendamento de prova substitutiva' (+1 variante(s)) — 409 chars  [redirecionado]
  - 'Matrícula em disciplina isolada' (+2 variante(s)) — 366 chars  [redirecionado]
  - 'Datas do ca

## Validação

In [5]:
_existentes = set(
    pd.concat([df_modelos["assunto"], df_modelos["assuntos_variantes"]])
    .dropna().str.casefold()
)

# `slots`/`assuntos_variantes` aceitam string pronta ('; '-separada, como o
# arquivo guarda) OU lista de strings (mais natural de escrever à mão — sem
# risco de esquecer o separador). `_texto_ou_lista` normaliza os dois formatos
# na célula de geração; aqui só valida que o tipo é um dos dois aceitos.
#
# `assunto`/`texto_modelo` continuam exigindo string pura: são texto livre,
# não uma lista de itens, e o erro mais comum ali é outro — uma vírgula
# sobrando dentro dos parênteses de uma string multi-linha vira uma tupla de
# 1 item (`AttributeError: 'tuple' object has no attribute 'strip'` sem essa
# checagem).
_CAMPOS_TEXTO = ("assunto", "texto_modelo")
_CAMPOS_TEXTO_OU_LISTA = ("slots", "assuntos_variantes")


def _tipo_valido_texto_ou_lista(valor):
    if valor is None or isinstance(valor, str):
        return True
    return isinstance(valor, (list, tuple)) and all(isinstance(v, str) for v in valor)


for i, m in enumerate(NOVOS_MODELOS):
    rotulo = f"NOVOS_MODELOS[{i}]"
    for campo in _CAMPOS_TEXTO:
        valor = m.get(campo)
        if not isinstance(valor, str):
            raise TypeError(
                f"{rotulo}['{campo}'] é {type(valor).__name__} ({valor!r}), esperava string. "
                "Causa comum: uma vírgula sobrando dentro dos parênteses de uma string "
                "multi-linha vira uma tupla de 1 item — confira se não tem `,` depois "
                "da última linha do texto, logo antes do `)`."
            )
    for campo in _CAMPOS_TEXTO_OU_LISTA:
        valor = m.get(campo)
        if not _tipo_valido_texto_ou_lista(valor):
            raise TypeError(
                f"{rotulo}['{campo}'] é {type(valor).__name__} ({valor!r}), esperava string, "
                "lista de strings, ou None. No arquivo este campo vira uma única string "
                "separada por '; ' — passe `\"a; b; c\"` ou `[\"a\", \"b\", \"c\"]`, os dois "
                "formatos funcionam (uma lista com item que não é string continua sendo erro)."
            )
    assert (m.get("assunto") or "").strip(), f"{rotulo}: 'assunto' vazio"
    assert (m.get("texto_modelo") or "").strip(), f"{rotulo}: 'texto_modelo' vazio"
    if m["assunto"].casefold() in _existentes:
        print(f"AVISO {rotulo}: já existe um modelo com assunto parecido — "
              f"confira se não é duplicata antes de seguir: {m['assunto']!r}")

print(f"{len(NOVOS_MODELOS)} modelo(s) validado(s) — sem erro bloqueante.")

AVISO NOVOS_MODELOS[12]: já existe um modelo com assunto parecido — confira se não é duplicata antes de seguir: 'Nota de corte'
13 modelo(s) validado(s) — sem erro bloqueante.


## Gera as linhas (mesma lógica de `ler_dados_pst.ipynb`, seção 6a)

In [6]:
def _texto_ou_lista(valor):
    """`slots`/`assuntos_variantes` -> a forma que o arquivo guarda: uma
    única string separada por "; ". Aceita já vir pronta (string) ou como
    lista (mais natural de escrever à mão) — ver checagem de tipo na célula
    de Validação, que roda antes desta."""
    if valor is None:
        return None
    if isinstance(valor, str):
        return valor.strip() or None
    return "; ".join(v.strip() for v in valor if v.strip()) or None


# Relê do disco NA HORA — não confia no `df_modelos`/`df_chunks` de "Estado
# atual do arquivo", que pode estar desatualizado se essa célula não foi
# reexecutada depois de uma gravação anterior nesta mesma sessão do kernel.
# Foi exatamente isso que gerou uma inconsistência real neste arquivo uma
# vez: duas linhas em `chunks` sem par em `modelos`, com `assunto` que nem
# batia com o `texto` — sintoma de `modelo_id` calculado sobre estado velho
# enquanto o corpo do texto já era de uma edição mais nova.
df_modelos = pd.read_excel(CAMINHO_XLSX, sheet_name="modelos")
df_chunks = pd.read_excel(CAMINHO_XLSX, sheet_name="chunks")

_orfaos = set(df_chunks["modelo_id"]) - set(df_modelos["modelo_id"])
if _orfaos:
    raise RuntimeError(
        f"Arquivo inconsistente: {sorted(_orfaos)} aparece em 'chunks' mas não "
        "em 'modelos'. Resolva isso antes de adicionar mais um modelo — senão "
        "o próximo modelo_id pode colidir com um chunk órfão. Causa mais "
        "provável: uma gravação anterior nesta sessão usou dado desatualizado "
        "(reexecute o notebook do início, célula por célula, em vez de só as "
        "de baixo). Se não for isso, confira manualmente as duas abas."
    )

# Maior modelo_id em QUALQUER uma das duas abas — não só em `modelos`. Cobre
# o caso (raro, mas já aconteceu) de `chunks` ter avançado mais que `modelos`
# sem virar órfão detectável pela checagem acima (ex.: um modelo_id que
# existe nas duas abas, mas `chunks` tem um número MAIOR que `modelos` seguiu
# usando).
_max_num = max(
    df_modelos["modelo_id"].str.slice(3).astype(int).max() if len(df_modelos) else 0,
    df_chunks["modelo_id"].str.slice(3).astype(int).max() if len(df_chunks) else 0,
)

agora = pd.Timestamp(datetime.now())
proximo_num = _max_num + 1

novas_linhas_modelos = []
novas_linhas_chunks = []

for m in NOVOS_MODELOS:
    modelo_id = f"MOD{proximo_num:04d}"
    proximo_num += 1

    assunto = m["assunto"].strip()
    slots = _texto_ou_lista(m.get("slots"))
    texto_modelo = m["texto_modelo"].strip()
    assuntos_variantes = _texto_ou_lista(m.get("assuntos_variantes")) or assunto
    redirecionado = bool(m.get("redirecionado", False))
    n_ocorrencias = int(m.get("n_ocorrencias", 1))
    n_threads = int(m.get("n_threads", 1))

    novas_linhas_modelos.append({
        "modelo_id": modelo_id, "assunto": assunto, "n_ocorrencias": n_ocorrencias,
        "n_threads": n_threads, "primeira_data": agora, "ultima_data": agora,
        "slots": slots, "assuntos_variantes": assuntos_variantes,
        "thread_ids": "",  # sem thread real: modelo escrito à mão, não extraído de e-mail
        "texto_modelo": texto_modelo, "redirecionado": redirecionado,
    })

    # --- a partir daqui, a MESMA lógica de ler_dados_pst.ipynb, seção 6a ---
    cabec = (f"Modelo de resposta padrao do suporte "
             f"(usado {n_ocorrencias}x em {n_threads} thread{'s' if n_threads != 1 else ''}).")
    corpo = (f"Assunto tipico: {assunto}\n"
             + (f"Campos a preencher: {slots}\n" if slots else "")
             + f"\n{texto_modelo}")
    ano = agora.year

    for i, ch in enumerate(dividir(corpo), start=1):
        texto = f"{cabec}\n\n{ch}"
        novas_linhas_chunks.append({
            "chunk_id": f"{modelo_id}-{i:03d}", "modelo_id": modelo_id, "tipo": "modelo",
            "parte": i, "assunto": assunto, "assuntos_variantes": assuntos_variantes,
            "slots": slots, "redirecionado": redirecionado,
            "n_ocorrencias": n_ocorrencias, "n_threads": n_threads,
            "primeira_data": agora, "ultima_data": agora, "ano": ano,
            "n_chars": len(texto), "tokens_aprox": round(len(texto) / 4), "texto": texto,
        })

print(f"{len(novas_linhas_modelos)} modelo(s) -> {len(novas_linhas_chunks)} chunk(s) novo(s)")
pd.DataFrame(novas_linhas_chunks)[["chunk_id", "parte", "n_chars", "tokens_aprox"]]

13 modelo(s) -> 13 chunk(s) novo(s)


,chunk_id,parte,n_chars,tokens_aprox
0,MOD0185-001,1,616,154
1,MOD0186-001,1,677,169
2,MOD0187-001,1,599,150
3,MOD0188-001,1,559,140
4,MOD0189-001,1,610,152
5,MOD0190-001,1,557,139
6,MOD0191-001,1,773,193
7,MOD0192-001,1,619,155
8,MOD0193-001,1,886,222
9,MOD0194-001,1,549,137


## Confira antes de gravar

In [7]:
for linha in novas_linhas_chunks:
    print("=" * 70)
    print(linha["chunk_id"], "-", linha["n_chars"], "chars,", linha["tokens_aprox"], "tokens aprox.")
    print(linha["texto"])

MOD0185-001 - 616 chars, 154 tokens aprox.
Modelo de resposta padrao do suporte (usado 1x em 1 thread).

Assunto tipico: Alteração de e-mail cadastrado
Campos a preencher: nome

Olá, {{nome}}!

Em relação ao seu e-mail pessoal cadastrado, pedimos, por gentileza, que entre em contato diretamente com o PUC Digital pelo e-mail puc-digital@puc-campinas.edu.br para solicitar a alteração.

Caso a solicitação seja referente ao seu e-mail institucional, informamos que não é possível realizar a alteração, pois esse endereço é gerado automaticamente a partir do seu RA (Registro Acadêmico).

Agradecemos a compreensão e permanecemos à disposição.

Atenciosamente,
MOD0186-001 - 677 chars, 169 tokens aprox.
Modelo de resposta padrao do suporte (usado 1x em 1 thread).

Assunto tipico: Envio de atividade no Canvas com prazo expirado
Campos a preencher: nome

Olá, {{nome}}!

O Canvas só permite o envio após a data limite quando o professor mantém a atividade aberta para envios atrasados. Se o botão de 

## Grava (com backup)

Sobrescreve `modelos_resposta_chunks.xlsx` com o conteúdo antigo + as linhas
novas. Backup do arquivo anterior vai para `.bak/` (ao lado deste notebook,
fora de `data/raw/` — ver comentário na 1ª célula de código).

In [8]:
PASTA_BACKUP.mkdir(exist_ok=True)
carimbo = datetime.now().strftime("%Y%m%dT%H%M%S")
destino_backup = PASTA_BACKUP / f"{CAMINHO_XLSX.stem}_{carimbo}{CAMINHO_XLSX.suffix}"
shutil.copy2(CAMINHO_XLSX, destino_backup)
print("backup:", destino_backup)

df_modelos_final = pd.concat([df_modelos, pd.DataFrame(novas_linhas_modelos)], ignore_index=True)
df_chunks_final = pd.concat([df_chunks, pd.DataFrame(novas_linhas_chunks)], ignore_index=True)

# Mesma ordem de coluna do arquivo original — o loader de ingestão não depende
# disso, mas mantém previsível para quem abrir no Excel.
df_modelos_final = df_modelos_final[df_modelos.columns]
df_chunks_final = df_chunks_final[df_chunks.columns]

with pd.ExcelWriter(CAMINHO_XLSX, engine="openpyxl") as xw:
    df_chunks_final.to_excel(xw, sheet_name="chunks", index=False)
    df_modelos_final.to_excel(xw, sheet_name="modelos", index=False)

print(f"gravado: {CAMINHO_XLSX}")
print(f"modelos: {len(df_modelos)} -> {len(df_modelos_final)}")
print(f"chunks : {len(df_chunks)} -> {len(df_chunks_final)}")

backup: C:\Users\henrique.cordeiro\Desktop\PPP\ia-agent-puc-digital\scripts\Ler_dados_exportados_email\.bak\modelos_resposta_chunks_20260910T132942.xlsx
gravado: C:\Users\henrique.cordeiro\Desktop\PPP\ia-agent-puc-digital\data\raw\email_modelos\modelos_resposta_chunks.xlsx
modelos: 184 -> 197
chunks : 203 -> 216


## Próximo passo (fora do notebook)

```bash
python -m scripts.ingest email_modelos
```

Reindexa o arquivo inteiro (ver aviso no topo — não é incremental). Se algo
saiu errado, o arquivo anterior está em `.bak/` ao lado deste notebook.